# 🛒 E-Commerce Customer Data Analytics Project

**Author:** Suptashree Rout  
**Program:** IBM SkillsBuild Data Analytics with AI Internship  
**Dataset:** customer_master-selected-columns.csv  

---

## Project Objective
This notebook performs a comprehensive data analytics and AI study on an e-commerce customer master dataset.  
The analysis covers:
- Data exploration and quality checks
- Demographic and geographic analysis
- Customer segmentation analysis
- K-Means Clustering (AI/ML)
- Business insights and recommendations

---

## 📦 Section 1: Import Libraries

In [ ]:
# Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Plot Styling
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('✅ All libraries imported successfully!')

---
## 📂 Section 2: Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('customer_master-selected-columns.csv')
print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')

In [ ]:
# Preview first 5 rows
print('--- First 5 rows ---')
df.head()

In [ ]:
# Last 5 rows
print('--- Last 5 rows ---')
df.tail()

In [ ]:
# Dataset Info
print('--- Dataset Information ---')
df.info()

In [ ]:
# Statistical Summary
print('--- Statistical Summary ---')
df.describe(include='all')

In [ ]:
# Column names
print('Column Names:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i}. {col}')

---
## 🔍 Section 3: Data Quality Check

In [ ]:
# Missing Values
print('--- Missing Values per Column ---')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Duplicate Records
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

# Unique counts per categorical column
print('\n--- Unique Value Counts ---')
cat_cols = ['gender', 'customer_segment', 'customer_country', 'region']
for col in cat_cols:
    print(f'{col}: {df[col].nunique()} unique values → {df[col].unique().tolist()}')

In [ ]:
# Age range check
print(f'Age range: {df["customer_age"].min()} to {df["customer_age"].max()}')
print(f'Mean age: {df["customer_age"].mean():.1f}')
print(f'Median age: {df["customer_age"].median()}')

---
## 📊 Section 4: Univariate Analysis

In [ ]:
# 4.1 Age Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['customer_age'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Customer Age Distribution (Histogram)')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Number of Customers')
axes[0].axvline(df['customer_age'].mean(), color='red', linestyle='--', label=f'Mean: {df["customer_age"].mean():.1f}')
axes[0].legend()

# KDE Plot
sns.kdeplot(df['customer_age'], ax=axes[1], fill=True, color='coral', linewidth=2)
axes[1].set_title('Customer Age Distribution (KDE Density)')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Density')

plt.suptitle('Customer Age Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('📌 Insight: Customer ages are fairly uniformly distributed between ~18–80, with a slight concentration in 30–60 age group.')

In [ ]:
# 4.2 Gender Distribution
gender_counts = df['gender'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar Chart
bars = axes[0].bar(gender_counts.index, gender_counts.values, color=['#2196F3', '#E91E63'], edgecolor='white', alpha=0.9)
axes[0].set_title('Gender Distribution (Count)')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of Customers')
for bar, val in zip(bars, gender_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, str(val), ha='center', fontweight='bold')

# Pie Chart
axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#2196F3', '#E91E63'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Gender Distribution (Percentage)')

plt.suptitle('Gender Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'📌 Insight: Female customers ({gender_counts.get("Female", 0):,}) slightly outnumber Male customers ({gender_counts.get("Male", 0):,}).')

In [ ]:
# 4.3 Customer Segment Distribution
seg_counts = df['customer_segment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = sns.color_palette('Set2', len(seg_counts))

# Horizontal Bar
bars = axes[0].barh(seg_counts.index, seg_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Customer Segment Distribution')
axes[0].set_xlabel('Number of Customers')
for bar, val in zip(bars, seg_counts.values):
    axes[0].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, f'{val:,} ({val/len(df)*100:.1f}%)',
                 va='center', fontsize=10)

# Pie
axes[1].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Segment Share (%)')

plt.suptitle('Customer Segment Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'📌 Insight: Consumer segment is the largest ({seg_counts.iloc[0]:,} customers), followed by VIP, Premium, and Business.')

---
## 🌍 Section 5: Geographic Analysis

In [ ]:
# 5.1 Top 10 Countries
top_countries = df['customer_country'].value_counts().head(10)

plt.figure(figsize=(11, 5))
bars = plt.bar(top_countries.index, top_countries.values,
               color=sns.color_palette('Blues_d', len(top_countries)), edgecolor='white')
for bar, val in zip(bars, top_countries.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
             f'{val:,}', ha='center', fontsize=9, fontweight='bold')
plt.title('Top 10 Countries by Customer Count', fontsize=14, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Number of Customers')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(f'📌 Insight: USA is the dominant market with {top_countries.iloc[0]:,} customers, followed by Germany and UK.')

In [ ]:
# 5.2 Top 10 States
top_states = df['customer_state'].value_counts().head(10)

plt.figure(figsize=(12, 5))
bars = plt.barh(top_states.index[::-1], top_states.values[::-1],
                color=sns.color_palette('Greens_d', len(top_states)), edgecolor='white')
for bar, val in zip(bars, top_states.values[::-1]):
    plt.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=9, fontweight='bold')
plt.title('Top 10 States by Customer Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Customers')
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Regional Distribution
region_counts = df['region'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

region_colors = sns.color_palette('Paired', len(region_counts))

axes[0].bar(region_counts.index, region_counts.values, color=region_colors, edgecolor='white')
axes[0].set_title('Customer Count by Region')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Number of Customers')
for i, (idx, val) in enumerate(region_counts.items()):
    axes[0].text(i, val + 30, f'{val:,}', ha='center', fontsize=9, fontweight='bold')

axes[1].pie(region_counts.values, labels=region_counts.index, autopct='%1.1f%%',
            colors=region_colors, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Regional Distribution (%)')

plt.suptitle('Geographic Region Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('📌 Insight: Customers are broadly spread across all regions with South and East slightly ahead.')

---
## 📈 Section 6: Bivariate & Multivariate Analysis

In [ ]:
# 6.1 Age Distribution by Customer Segment (Box Plot)
plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x='customer_segment', y='customer_age',
            palette='Set2', order=df['customer_segment'].value_counts().index)
plt.title('Age Distribution by Customer Segment', fontsize=14, fontweight='bold')
plt.xlabel('Customer Segment')
plt.ylabel('Age')
plt.tight_layout()
plt.show()
print('📌 Insight: All segments have a broadly similar age spread (IQR ~30–60). VIP and Business show slightly higher medians.')

In [ ]:
# 6.2 Gender Distribution across Customer Segments (Grouped Bar)
seg_gender = df.groupby(['customer_segment', 'gender']).size().unstack(fill_value=0)

ax = seg_gender.plot(kind='bar', figsize=(11, 5), color=['#2196F3', '#E91E63'],
                     edgecolor='white', alpha=0.9)
plt.title('Gender Distribution across Customer Segments', fontsize=14, fontweight='bold')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.legend(title='Gender')
plt.tight_layout()
plt.show()
print('📌 Insight: Female customers outnumber Male customers in most segments, especially Consumer and Premium.')

In [ ]:
# 6.3 Segment distribution across Regions (Heatmap)
pivot = df.groupby(['region', 'customer_segment']).size().unstack(fill_value=0)

plt.figure(figsize=(11, 5))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            linecolor='white', cbar_kws={'label': 'Customer Count'})
plt.title('Customer Segment Distribution by Region (Heatmap)', fontsize=14, fontweight='bold')
plt.xlabel('Customer Segment')
plt.ylabel('Region')
plt.tight_layout()
plt.show()
print('📌 Insight: Consumer segment is uniformly strong across all regions. South region has the highest absolute consumer counts.')

In [ ]:
# 6.4 Age Distribution by Country (Top 5 countries – Violin Plot)
top5_countries = df['customer_country'].value_counts().head(5).index
df_top5 = df[df['customer_country'].isin(top5_countries)]

plt.figure(figsize=(12, 5))
sns.violinplot(data=df_top5, x='customer_country', y='customer_age',
               palette='muted', inner='quartile',
               order=top5_countries)
plt.title('Age Distribution by Top 5 Countries', fontsize=14, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Age')
plt.tight_layout()
plt.show()
print('📌 Insight: All top countries show a similar bimodal age distribution pattern, indicating globally consistent customer demographics.')

In [ ]:
# 6.5 Age groups analysis
bins = [0, 25, 35, 45, 55, 65, 100]
labels = ['18–25', '26–35', '36–45', '46–55', '56–65', '65+']
df['age_group'] = pd.cut(df['customer_age'], bins=bins, labels=labels, right=True)

age_group_counts = df['age_group'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
bars = plt.bar(age_group_counts.index.astype(str), age_group_counts.values,
               color=sns.color_palette('viridis', len(age_group_counts)), edgecolor='white')
for bar, val in zip(bars, age_group_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{val:,}', ha='center', fontsize=9, fontweight='bold')
plt.title('Customer Count by Age Group', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()
print('📌 Insight: The 36–45 and 46–55 age groups are the largest, forming the core e-commerce customer base.')

---
## 🤖 Section 7: AI/ML — Customer Segmentation with K-Means Clustering

In [ ]:
# 7.1 Feature Engineering
print('--- Preparing features for K-Means Clustering ---')

# Encode categorical features
le_gender = LabelEncoder()
le_segment = LabelEncoder()
le_region = LabelEncoder()

df_ml = df[['customer_age', 'gender', 'customer_segment', 'region']].copy()
df_ml['gender_enc'] = le_gender.fit_transform(df_ml['gender'])
df_ml['segment_enc'] = le_segment.fit_transform(df_ml['customer_segment'])
df_ml['region_enc'] = le_region.fit_transform(df_ml['region'])

# Select features for clustering
features = ['customer_age', 'gender_enc', 'segment_enc', 'region_enc']
X = df_ml[features]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix shape: {X_scaled.shape}')
print(f'Features used: {features}')
print('✅ Data prepared for clustering!')

In [ ]:
# 7.2 Elbow Method — Finding Optimal Number of Clusters
print('--- Running Elbow Method (K = 1 to 10) ---')

inertia = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_range, inertia, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Optimal K = 4')
plt.title('Elbow Method — Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-cluster Sum of Squares)')
plt.legend()
plt.xticks(k_range)
plt.tight_layout()
plt.show()
print('📌 Insight: The elbow bends at K=4, indicating 4 is the optimal number of clusters.')

In [ ]:
# 7.3 Apply K-Means with K=4
print('--- Applying K-Means Clustering with K=4 ---')

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print('Cluster distribution:')
cluster_dist = df['cluster'].value_counts().sort_index()
for cluster_id, count in cluster_dist.items():
    print(f'  Cluster {cluster_id}: {count:,} customers ({count/len(df)*100:.1f}%)')

In [ ]:
# 7.4 Cluster Profile Analysis
print('--- Cluster Profile Summary ---')
cluster_profile = df.groupby('cluster').agg(
    count=('customer_id', 'count'),
    avg_age=('customer_age', 'mean'),
    top_segment=('customer_segment', lambda x: x.mode()[0]),
    top_region=('region', lambda x: x.mode()[0]),
    top_gender=('gender', lambda x: x.mode()[0]),
    top_country=('customer_country', lambda x: x.mode()[0])
).round(1)
print(cluster_profile.to_string())

In [ ]:
# 7.5 Visualize Clusters — Age vs Segment
plt.figure(figsize=(11, 6))
scatter = plt.scatter(df['customer_age'],
                      df_ml['segment_enc'],
                      c=df['cluster'],
                      cmap='Set1',
                      alpha=0.4,
                      s=8)
plt.colorbar(scatter, label='Cluster ID')
plt.title('K-Means Clusters: Age vs Customer Segment', fontsize=14, fontweight='bold')
plt.xlabel('Customer Age')
plt.ylabel('Customer Segment (Encoded)')
plt.yticks(range(len(le_segment.classes_)), le_segment.classes_)
plt.tight_layout()
plt.show()

In [ ]:
# 7.6 Cluster Segment Composition (Stacked Bar)
cluster_seg = df.groupby(['cluster', 'customer_segment']).size().unstack(fill_value=0)

ax = cluster_seg.plot(kind='bar', stacked=True, figsize=(11, 5),
                      colormap='Set2', edgecolor='white', alpha=0.9)
plt.title('Customer Segment Composition per Cluster', fontsize=14, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.legend(title='Segment', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()
print('📌 Insight: Each cluster has a distinct segment composition, validating the effectiveness of K-Means segmentation.')

In [ ]:
# 7.7 Average Age per Cluster (Bar Chart)
avg_age_cluster = df.groupby('cluster')['customer_age'].mean().round(1)

plt.figure(figsize=(8, 4))
bars = plt.bar(avg_age_cluster.index.astype(str), avg_age_cluster.values,
               color=sns.color_palette('Set1', 4), edgecolor='white')
for bar, val in zip(bars, avg_age_cluster.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
plt.title('Average Customer Age per Cluster', fontsize=13, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Average Age')
plt.ylim(0, max(avg_age_cluster.values) + 5)
plt.tight_layout()
plt.show()

---
## 📋 Section 8: Summary Statistics Table

In [ ]:
# Final Summary
print('=' * 55)
print('         E-COMMERCE CUSTOMER DATASET — SUMMARY')
print('=' * 55)
print(f'  Total Customers         : {len(df):,}')
print(f'  Age Range               : {df["customer_age"].min()} – {df["customer_age"].max()}')
print(f'  Mean Age                : {df["customer_age"].mean():.1f}')
print(f'  Countries               : {df["customer_country"].nunique()}')
print(f'  States                  : {df["customer_state"].nunique()}')
print(f'  Unique Cities           : {df["customer_city"].nunique()}')
print(f'  Customer Segments       : {df["customer_segment"].nunique()} ({list(df["customer_segment"].unique())})')
print(f'  Top Country             : {df["customer_country"].value_counts().index[0]}')
print(f'  Top Segment             : {df["customer_segment"].value_counts().index[0]}')
print(f'  Top Region              : {df["region"].value_counts().index[0]}')
print(f'  Gender (Female)         : {(df["gender"]=="Female").sum():,} ({(df["gender"]=="Female").mean()*100:.1f}%)')
print(f'  Gender (Male)           : {(df["gender"]=="Male").sum():,} ({(df["gender"]=="Male").mean()*100:.1f}%)')
print(f'  K-Means Clusters        : 4')
print('=' * 55)

---
## 💡 Section 9: AI Insights & Business Recommendations

In [ ]:
insights = """
╔══════════════════════════════════════════════════════════════════╗
║          KEY BUSINESS INSIGHTS & RECOMMENDATIONS                ║
╚══════════════════════════════════════════════════════════════════╝

📌 INSIGHT 1 — Consumer Segment Dominance
   The Consumer segment accounts for ~50% of all customers.
   → Recommendation: Develop mass-market campaigns, loyalty programs,
     and discount bundles to retain and grow this core segment.

📌 INSIGHT 2 — USA Market Leadership
   USA is the #1 market by customer count.
   → Recommendation: Invest in US-specific personalization,
     localization, and express shipping to capture more market share.

📌 INSIGHT 3 — VIP & Premium Potential
   VIP and Premium segments are high-value but smaller in count.
   → Recommendation: Use targeted upsell and exclusive offers
     to convert Consumer customers into VIP/Premium tier.

📌 INSIGHT 4 — Female Customer Majority
   Female customers form the slight majority in most segments.
   → Recommendation: Design female-centric product curation,
     influencer partnerships, and personalized recommendations.

📌 INSIGHT 5 — 36–55 Core Age Demographic
   The 36–55 age band is the biggest customer cohort.
   → Recommendation: Target ads and content at middle-aged
     professionals with family-oriented and convenience-based messaging.

📌 INSIGHT 6 — K-Means Cluster Actionability
   4 distinct clusters emerge from age + segment + region + gender.
   → Recommendation: Use cluster IDs to power a recommendation engine
     (collaborative filtering) for personalized product suggestions.

📌 INSIGHT 7 — Geographic Expansion Opportunity
   India and Canada show growing customer presence.
   → Recommendation: Invest in localized storefronts and
     local payment methods for India and Canada markets.
"""
print(insights)

---
## ✅ Section 10: Conclusion

### Project Conclusion

This project successfully demonstrated a complete **end-to-end Data Analytics and AI pipeline** on a real-world e-commerce customer dataset.

**What was accomplished:**
- ✅ Loaded and explored 25,001 customer records across 10 features
- ✅ Performed thorough data quality checks (missing values, duplicates, types)
- ✅ Conducted univariate, bivariate, and multivariate visual analysis
- ✅ Mapped customer geographic distribution across countries, states, and regions
- ✅ Applied **K-Means Clustering** to identify 4 distinct customer groups
- ✅ Generated actionable business insights and strategic recommendations

**Technologies Used:**  
`Python` | `Pandas` | `NumPy` | `Matplotlib` | `Seaborn` | `Scikit-learn` | `Jupyter Notebook`

---
*Project by **Suptashree Rout** — IBM SkillsBuild Data Analytics with AI Internship*